# Notebook 14: Handling Imbalanced Classification

## 📚 Historical Context

**Timeline**: 1990s-2023 - From ignoring the problem to specialized solutions

**The Evolution of Handling Imbalance**:
- **1990s-2000**: Academic awareness
  - Problem: Standard accuracy metric misleading
  - Example: 99% "not fraud" → 99% accuracy but catches 0% fraud
  - Most practitioners ignored the issue

- **2002**: SMOTE (Synthetic Minority Over-sampling Technique)
  - Chawla et al. introduce data augmentation for minority class
  - Generate synthetic examples by interpolation
  - Widely adopted in industry

- **2006**: Cost-Sensitive Learning
  - Assign different costs to different types of errors
  - Medical: Missing disease (false negative) worse than false alarm
  - Fraud: Missing fraud worse than investigating legitimate transaction

- **2017**: Focal Loss for Object Detection
  - 99.9% of pixels are background in object detection
  - RetinaNet paper shows focal loss solves extreme imbalance
  - Enables single-stage detectors to match two-stage

- **2020+**: Modern era - Combination approaches
  - Data-level: Sampling, augmentation
  - Algorithm-level: Loss functions, ensemble
  - Evaluation-level: Proper metrics (F1, AUC-PR)

**Real-World Examples**:

**Medical Diagnosis**:
```
Disease prevalence: 1%
99% healthy, 1% sick
Model predicts "healthy" for everyone → 99% accurate but USELESS
```

**Fraud Detection**:
```
Fraud rate: 0.1%
99.9% legitimate, 0.1% fraud
Missing fraud costs $1000, false alarm costs $1
Need to optimize for recall, not accuracy
```

**Manufacturing Defects**:
```
Defect rate: 0.5%
99.5% good, 0.5% defective
Missing defects damages reputation
False alarms waste inspection time
```

## 🎯 What You'll Learn

1. **Problem Setup** - Binary imbalanced classification (95/5 split)
2. **Evaluation Metrics** - Why accuracy is misleading, proper metrics
3. **Baseline** - Standard approach failure
4. **Class Weights** - Simple but effective
5. **Focal Loss** - For extreme imbalance
6. **Threshold Tuning** - Optimize for your metric
7. **Comprehensive Comparison** - Which method works best when

## 💡 Why This Matters

Imbalanced classification is EVERYWHERE in real-world ML:
- **95% of ML projects have some degree of imbalance**
- **Accuracy is almost always the wrong metric**
- **Default training = model ignores minority class**
- **Wrong approach = deployment disaster**

**Story time**: A company deployed a fraud detection model with "99.9% accuracy". It caught ZERO fraud. Why? Because fraud was 0.1% of transactions, and the model learned to predict "not fraud" for everything. Cost: $10M in missed fraud. This notebook teaches you how to avoid this.

---

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Problem Setup - Creating Imbalanced Dataset

### Scenario: Credit Card Fraud Detection

**Dataset characteristics**:
- 95% legitimate transactions (class 0)
- 5% fraudulent transactions (class 1)
- 30 features (anonymized transaction details)
- Binary classification problem

**Business constraints**:
- Missing fraud: Loses money
- False alarm: Annoys customer
- Cost of missing fraud >> cost of false alarm

### Key Terminology:

**Positive class (1)**: The rare class we care about (fraud)
**Negative class (0)**: The common class (legitimate)

**Confusion Matrix**:
```
                 Predicted
              |  0  |  1  |
        ------+-----+-----+
Actual   0    | TN  | FP  |  ← Legitimate
         1    | FN  | TP  |  ← Fraud
```

**Metrics**:
- **Accuracy**: (TP + TN) / (TP + TN + FP + FN) ← MISLEADING FOR IMBALANCED
- **Precision**: TP / (TP + FP) ← Of predicted fraud, how many are real?
- **Recall**: TP / (TP + FN) ← Of real fraud, how many did we catch?
- **F1**: 2 * (Precision * Recall) / (Precision + Recall) ← Harmonic mean

### Why Accuracy Fails:

**Example with 95/5 split**:
```
Total: 1000 samples
Legitimate: 950
Fraud: 50

Dumb model: Always predict "legitimate"
TP=0, TN=950, FP=0, FN=50
Accuracy = 950/1000 = 95%  ← Looks great!
Recall = 0/50 = 0%  ← Actually terrible!
```

In [ ]:
class ImbalancedBinaryDataset(Dataset):
    """
    Binary classification dataset with severe imbalance.
    
    Simulates fraud detection: 95% legitimate, 5% fraud.
    """
    def __init__(self, num_samples=10000, minority_ratio=0.05, num_features=30, seed=42):
        """
        Args:
            num_samples: Total number of samples
            minority_ratio: Ratio of minority class (fraud)
            num_features: Feature dimension
            seed: Random seed
        """
        np.random.seed(seed)
        torch.manual_seed(seed)
        
        self.num_samples = num_samples
        self.num_features = num_features
        
        # Calculate samples per class
        num_minority = int(num_samples * minority_ratio)
        num_majority = num_samples - num_minority
        
        # Generate features with different distributions
        # Majority class (legitimate): Normal distribution
        majority_features = np.random.randn(num_majority, num_features).astype(np.float32)
        majority_features *= 0.5  # Lower variance
        
        # Minority class (fraud): Shifted distribution
        # Make it learnable but not too easy
        minority_features = np.random.randn(num_minority, num_features).astype(np.float32)
        minority_features *= 0.7  # Slightly higher variance
        minority_features += 1.5   # Shift mean
        
        # Add some overlap (real-world is not perfectly separable)
        noise = np.random.randn(num_minority, num_features).astype(np.float32) * 0.3
        minority_features += noise
        
        # Combine
        self.features = np.vstack([majority_features, minority_features])
        self.labels = np.hstack([
            np.zeros(num_majority, dtype=np.int64),
            np.ones(num_minority, dtype=np.int64)
        ])
        
        # Shuffle
        indices = np.random.permutation(num_samples)
        self.features = self.features[indices]
        self.labels = self.labels[indices]
        
        # Convert to tensors
        self.features = torch.from_numpy(self.features)
        self.labels = torch.from_numpy(self.labels)
        
        # Print statistics
        print(f"Dataset created:")
        print(f"  Total samples: {num_samples}")
        print(f"  Legitimate (0): {num_majority} ({num_majority/num_samples*100:.1f}%)")
        print(f"  Fraud (1):      {num_minority} ({num_minority/num_samples*100:.1f}%)")
        print(f"  Imbalance ratio: {num_majority/num_minority:.1f}:1")
        print(f"  Feature dimension: {num_features}")
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]
    
    def get_class_weights(self):
        """Compute class weights for weighted loss."""
        class_counts = Counter(self.labels.numpy())
        total = sum(class_counts.values())
        num_classes = len(class_counts)
        
        # Balanced class weights
        weights = {
            cls: total / (num_classes * count) 
            for cls, count in class_counts.items()
        }
        
        # Convert to tensor
        weight_tensor = torch.tensor(
            [weights[0], weights[1]], 
            dtype=torch.float32
        )
        return weight_tensor

# Create datasets
print("="*60)
print("CREATING IMBALANCED DATASET")
print("="*60)

train_dataset = ImbalancedBinaryDataset(
    num_samples=10000, 
    minority_ratio=0.05,
    seed=42
)

val_dataset = ImbalancedBinaryDataset(
    num_samples=2000,
    minority_ratio=0.05, 
    seed=123
)

test_dataset = ImbalancedBinaryDataset(
    num_samples=2000,
    minority_ratio=0.05,
    seed=456
)

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Class distribution
train_labels = train_dataset.labels.numpy()
class_counts = Counter(train_labels)
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], 
            [class_counts[0], class_counts[1]],
            color=['green', 'red'], alpha=0.7)
axes[0].set_ylabel('Number of Samples')
axes[0].set_title('Class Distribution (Training Set)')
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Feature distribution comparison
legitimate_features = train_dataset.features[train_labels == 0]
fraud_features = train_dataset.features[train_labels == 1]

# Plot first feature as example
axes[1].hist(legitimate_features[:, 0].numpy(), bins=50, alpha=0.6, 
             label='Legitimate', color='green')
axes[1].hist(fraud_features[:, 0].numpy(), bins=50, alpha=0.6,
             label='Fraud', color='red')
axes[1].set_xlabel('Feature 0 Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Feature Distribution (Sample Feature)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Note: Classes have different but overlapping distributions.")
print("This makes it learnable but challenging, like real fraud detection.")
print("="*60)

## Part 2: Evaluation Metrics Deep Dive

### Why Standard Metrics Fail:

**Accuracy**: Misleading for imbalanced data
```python
# Predicting all legitimate:
accuracy = 95%  ← High but useless
```

### Proper Metrics for Imbalanced Classification:

**1. Precision** (Positive Predictive Value):
```
Precision = TP / (TP + FP)
= "Of all predicted fraud, how many are actually fraud?"

High precision = Few false alarms
Use when: False positives are expensive
Example: Don't want to annoy legitimate customers
```

**2. Recall** (Sensitivity, True Positive Rate):
```
Recall = TP / (TP + FN)
= "Of all actual fraud, how many did we catch?"

High recall = Few missed fraud cases
Use when: False negatives are expensive
Example: Must catch as much fraud as possible
```

**3. F1 Score** (Harmonic Mean):
```
F1 = 2 * (Precision * Recall) / (Precision + Recall)
= Balanced metric between precision and recall

Use when: Both false positives and false negatives matter
```

**4. PR-AUC** (Precision-Recall Area Under Curve):
```
Area under precision-recall curve
Better than ROC-AUC for imbalanced data

Perfect classifier: PR-AUC = 1.0
Random classifier: PR-AUC ≈ minority class ratio
```

### Precision-Recall Trade-off:

**Can't maximize both simultaneously**:
```
Scenario 1: High threshold (predict fraud rarely)
  Precision: High (when we predict fraud, usually correct)
  Recall: Low (miss many fraud cases)
  
Scenario 2: Low threshold (predict fraud often)
  Precision: Low (many false alarms)
  Recall: High (catch most fraud)
  
Scenario 3: Optimal threshold (tune for your use case)
  Balance based on business costs
```

### ROC-AUC vs PR-AUC:

**ROC-AUC** (Receiver Operating Characteristic):
- Plots TPR (recall) vs FPR
- Good for balanced datasets
- Misleading for imbalanced (can look good even if useless)

**PR-AUC** (Precision-Recall):
- Plots Precision vs Recall
- Better for imbalanced datasets
- More sensitive to minority class performance

In [ ]:
def compute_metrics(y_true, y_pred, y_prob=None, print_report=True):
    """
    Compute comprehensive metrics for imbalanced classification.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
        y_prob: Predicted probabilities (for AUC)
        print_report: Whether to print classification report
    
    Returns:
        Dictionary of metrics
    """
    metrics = {}
    
    # Basic metrics
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
    metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    metrics['true_negatives'] = int(tn)
    metrics['false_positives'] = int(fp)
    metrics['false_negatives'] = int(fn)
    metrics['true_positives'] = int(tp)
    
    # Specificity (true negative rate)
    metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # AUC metrics (if probabilities provided)
    if y_prob is not None:
        # ROC-AUC
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        metrics['roc_auc'] = auc(fpr, tpr)
        
        # PR-AUC (better for imbalanced)
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        metrics['pr_auc'] = auc(recall, precision)
        metrics['avg_precision'] = average_precision_score(y_true, y_prob)
    
    if print_report:
        print("\nClassification Report:")
        print("-" * 60)
        print(classification_report(y_true, y_pred, 
                                    target_names=['Legitimate', 'Fraud'],
                                    digits=4))
        
        print("Confusion Matrix:")
        print(f"                Predicted")
        print(f"              Legit  Fraud")
        print(f"Actual Legit  {tn:5d}  {fp:5d}")
        print(f"       Fraud  {fn:5d}  {tp:5d}")
        print()
        
        print("Key Metrics:")
        print(f"  Accuracy:   {metrics['accuracy']:.4f}  ← Overall correctness")
        print(f"  Precision:  {metrics['precision']:.4f}  ← Of predicted fraud, how many correct?")
        print(f"  Recall:     {metrics['recall']:.4f}  ← Of actual fraud, how many caught?")
        print(f"  F1 Score:   {metrics['f1']:.4f}  ← Harmonic mean of precision and recall")
        
        if y_prob is not None:
            print(f"  ROC-AUC:    {metrics['roc_auc']:.4f}")
            print(f"  PR-AUC:     {metrics['pr_auc']:.4f}  ← Better for imbalanced data")
    
    return metrics

def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    """Plot confusion matrix with annotations."""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Legitimate', 'Fraud'],
                yticklabels=['Legitimate', 'Fraud'],
                cbar_kws={'label': 'Count'})
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Demonstrate why accuracy is misleading
print("\n" + "="*60)
print("DEMONSTRATION: WHY ACCURACY IS MISLEADING")
print("="*60)

# Simulate "predict all legitimate" baseline
val_labels = val_dataset.labels.numpy()
dummy_predictions = np.zeros_like(val_labels)  # Predict all 0

print("\nBaseline: Always predict 'Legitimate' (class 0)")
dummy_metrics = compute_metrics(val_labels, dummy_predictions, print_report=True)

print("\n" + "="*60)
print("INSIGHT:")
print(f"95% accuracy looks great, but recall is {dummy_metrics['recall']:.1%}!")
print("This model catches ZERO fraud cases.")
print("This is why we need proper metrics and techniques.")
print("="*60)

## Part 3: Model Architecture and Training Setup

### Simple Binary Classifier:

We'll use a simple feed-forward network to focus on the imbalance handling techniques, not model complexity.

**Architecture**:
```
Input (30 features)
  ↓
Linear(30 → 64) + ReLU + Dropout
  ↓
Linear(64 → 32) + ReLU + Dropout
  ↓
Linear(32 → 2) → Logits
  ↓
Softmax → Probabilities
```

### Training Function:

We'll create a reusable training function that can work with different loss functions and techniques.

In [ ]:
class BinaryClassifier(nn.Module):
    """Simple binary classifier for fraud detection."""
    def __init__(self, input_dim=30, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 2)
        )
    
    def forward(self, x):
        return self.network(x)

# Focal Loss implementation
class FocalLoss(nn.Module):
    """Focal loss for handling imbalance."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_loss = self.alpha * ((1 - p_t) ** self.gamma) * ce_loss
        return focal_loss.mean()

def train_model(
    model, 
    train_loader, 
    val_loader,
    criterion, 
    num_epochs=20,
    lr=1e-3,
    name="Model"
):
    """
    Train binary classifier and return predictions.
    
    Args:
        model: PyTorch model
        train_loader: Training DataLoader
        val_loader: Validation DataLoader  
        criterion: Loss function
        num_epochs: Number of training epochs
        lr: Learning rate
        name: Model name for logging
    
    Returns:
        model: Trained model
        history: Training history
        val_preds: Validation predictions
        val_probs: Validation probabilities
        val_labels: Validation true labels
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    history = {'train_loss': [], 'val_f1': [], 'val_recall': []}
    best_f1 = 0.0
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        scheduler.step()
        
        # Validation
        model.eval()
        val_preds_list = []
        val_labels_list = []
        val_probs_list = []
        
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(device)
                outputs = model(features)
                probs = F.softmax(outputs, dim=-1)
                _, preds = torch.max(outputs, 1)
                
                val_preds_list.append(preds.cpu())
                val_labels_list.append(labels)
                val_probs_list.append(probs[:, 1].cpu())  # Prob of class 1
        
        val_preds = torch.cat(val_preds_list).numpy()
        val_labels = torch.cat(val_labels_list).numpy()
        val_probs = torch.cat(val_probs_list).numpy()
        
        # Compute metrics
        val_f1 = f1_score(val_labels, val_preds, zero_division=0)
        val_recall = recall_score(val_labels, val_preds, zero_division=0)
        
        history['train_loss'].append(train_loss)
        history['val_f1'].append(val_f1)
        history['val_recall'].append(val_recall)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}: "
                  f"Loss={train_loss:.4f}, F1={val_f1:.4f}, Recall={val_recall:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
    
    print(f"Best F1: {best_f1:.4f}")
    
    return model, history, val_preds, val_probs, val_labels

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print("\n" + "="*60)
print("TRAINING SETUP READY")
print("="*60)
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: 128")
print("="*60)

## Part 4: Baseline - Standard Cross-Entropy

### Hypothesis:

Standard cross-entropy will perform poorly because:
1. Treats all classes equally
2. Majority class dominates gradient
3. Model learns to ignore minority class

### Expected Behavior:
- High accuracy (>90%)
- Low recall for minority class (<50%)
- Model biased toward predicting majority class

Let's verify this hypothesis.

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 1: BASELINE (Standard Cross-Entropy)")
print("="*60)

# Train with standard CE
model_baseline = BinaryClassifier().to(device)
criterion_baseline = nn.CrossEntropyLoss()

print("\nTraining with standard cross-entropy...")
model_baseline, history_baseline, preds_baseline, probs_baseline, labels_val = train_model(
    model_baseline, train_loader, val_loader, criterion_baseline, num_epochs=20, name="Baseline"
)

print("\n" + "="*60)
print("BASELINE RESULTS")
print("="*60)
metrics_baseline = compute_metrics(labels_val, preds_baseline, probs_baseline, print_report=True)

# Visualize
plot_confusion_matrix(labels_val, preds_baseline, "Baseline: Standard Cross-Entropy")

print("\n" + "="*60)
print("ANALYSIS:")
print(f"As expected, high accuracy ({metrics_baseline['accuracy']:.1%}) but")
print(f"low recall ({metrics_baseline['recall']:.1%}) for fraud detection.")
print("The model is biased toward predicting 'legitimate'.")
print("="*60)

## Part 5: Class Weights - Simple but Effective

### Approach:

**Weighted Cross-Entropy**:
```python
weight_class_0 = n_samples / (n_classes * n_samples_class_0)
weight_class_1 = n_samples / (n_classes * n_samples_class_1)
```

**Effect**:
- Minority class errors weighted higher
- Forces model to pay attention to rare class
- Simple to implement

**For our 95/5 split**:
```
n_samples = 10000
n_class_0 = 9500
n_class_1 = 500

weight_0 = 10000 / (2 * 9500) = 0.526
weight_1 = 10000 / (2 * 500) = 10.0

Ratio: 10.0 / 0.526 ≈ 19:1
→ Misclassifying fraud costs 19x more!
```

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 2: CLASS WEIGHTS")
print("="*60)

# Compute class weights
class_weights = train_dataset.get_class_weights().to(device)
print(f"\nClass weights: {class_weights.cpu().numpy()}")
print(f"Weight ratio (fraud/legit): {class_weights[1]/class_weights[0]:.1f}:1")

# Train with weighted CE
model_weighted = BinaryClassifier().to(device)
criterion_weighted = nn.CrossEntropyLoss(weight=class_weights)

print("\nTraining with weighted cross-entropy...")
model_weighted, history_weighted, preds_weighted, probs_weighted, _ = train_model(
    model_weighted, train_loader, val_loader, criterion_weighted, num_epochs=20, name="Weighted"
)

print("\n" + "="*60)
print("WEIGHTED CROSS-ENTROPY RESULTS")
print("="*60)
metrics_weighted = compute_metrics(labels_val, preds_weighted, probs_weighted, print_report=True)

plot_confusion_matrix(labels_val, preds_weighted, "Weighted Cross-Entropy")

# Compare with baseline
print("\n" + "="*60)
print("COMPARISON WITH BASELINE")
print("="*60)
print(f"Metric          Baseline    Weighted    Improvement")
print("-" * 60)
print(f"Recall:         {metrics_baseline['recall']:.4f}      "
      f"{metrics_weighted['recall']:.4f}      "
      f"{metrics_weighted['recall'] - metrics_baseline['recall']:+.4f}")
print(f"F1 Score:       {metrics_baseline['f1']:.4f}      "
      f"{metrics_weighted['f1']:.4f}      "
      f"{metrics_weighted['f1'] - metrics_baseline['f1']:+.4f}")
print(f"PR-AUC:         {metrics_baseline['pr_auc']:.4f}      "
      f"{metrics_weighted['pr_auc']:.4f}      "
      f"{metrics_weighted['pr_auc'] - metrics_baseline['pr_auc']:+.4f}")
print("="*60)

## Part 6: Focal Loss - For Extreme Imbalance

### Focal Loss Formula:
```
FL = -α * (1 - p_t)^γ * log(p_t)
```

### Advantages over Class Weights:
1. **Focuses on hard examples**: Down-weights easy examples dynamically
2. **Self-adjusting**: Automatically reduces focus as model improves
3. **Better for extreme imbalance**: Works well at >100:1 ratios

### Hyperparameters:
- **γ = 2**: Standard focusing strength
- **α = 0.25**: Weight for positive class

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 3: FOCAL LOSS")
print("="*60)

# Train with focal loss
model_focal = BinaryClassifier().to(device)
criterion_focal = FocalLoss(alpha=0.25, gamma=2.0)

print("\nTraining with focal loss (γ=2, α=0.25)...")
model_focal, history_focal, preds_focal, probs_focal, _ = train_model(
    model_focal, train_loader, val_loader, criterion_focal, num_epochs=20, name="Focal"
)

print("\n" + "="*60)
print("FOCAL LOSS RESULTS")
print("="*60)
metrics_focal = compute_metrics(labels_val, preds_focal, probs_focal, print_report=True)

plot_confusion_matrix(labels_val, preds_focal, "Focal Loss")

## Part 7: Threshold Tuning - Optimize for Your Metric

### The Problem:

**Default threshold = 0.5**:
```python
if prob_fraud > 0.5:
    predict_fraud
else:
    predict_legitimate
```

But 0.5 is **arbitrary**! It may not be optimal for your metric or business constraint.

### Threshold Tuning:

**Grid search over thresholds**:
```python
for threshold in [0.1, 0.2, 0.3, ..., 0.9]:
    predictions = (probabilities > threshold)
    compute_metrics(predictions)
    
Choose threshold that maximizes your target metric
```

### Example Use Cases:

**Maximize Recall** (catch as much fraud as possible):
```
Use low threshold (e.g., 0.2)
→ More false alarms, but catch more fraud
```

**Maximize Precision** (minimize false alarms):
```
Use high threshold (e.g., 0.8)
→ Fewer alerts, but more likely to be real fraud
```

**Maximize F1** (balance both):
```
Find optimal threshold empirically
Often around 0.3-0.5 for imbalanced data
```

### Business-Driven Thresholds:

**Cost-based**:
```
Cost_FN = $1000  (missed fraud)
Cost_FP = $5     (false alarm)

Optimal threshold minimizes:
Total_Cost = FN * Cost_FN + FP * Cost_FP
```

In [ ]:
def find_optimal_threshold(y_true, y_prob, metric='f1'):
    """
    Find optimal classification threshold.
    
    Args:
        y_true: True labels
        y_prob: Predicted probabilities
        metric: 'f1', 'recall', or 'precision'
    
    Returns:
        best_threshold: Optimal threshold
        best_score: Best score achieved
        threshold_scores: Dict of threshold → score
    """
    thresholds = np.arange(0.05, 0.95, 0.05)
    scores = []
    
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        
        if metric == 'f1':
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == 'recall':
            score = recall_score(y_true, y_pred, zero_division=0)
        elif metric == 'precision':
            score = precision_score(y_true, y_pred, zero_division=0)
        else:
            raise ValueError(f"Unknown metric: {metric}")
        
        scores.append(score)
    
    best_idx = np.argmax(scores)
    best_threshold = thresholds[best_idx]
    best_score = scores[best_idx]
    
    threshold_scores = dict(zip(thresholds, scores))
    
    return best_threshold, best_score, threshold_scores

print("\n" + "="*60)
print("EXPERIMENT 4: THRESHOLD TUNING")
print("="*60)

# Use weighted model for threshold tuning
print("\nFinding optimal threshold for weighted CE model...")

# Find optimal for F1
best_f1_threshold, best_f1_score, f1_scores = find_optimal_threshold(
    labels_val, probs_weighted, metric='f1'
)

# Find optimal for Recall
best_recall_threshold, best_recall_score, recall_scores = find_optimal_threshold(
    labels_val, probs_weighted, metric='recall'
)

print(f"\nOptimal thresholds:")
print(f"  For F1:     threshold={best_f1_threshold:.2f}, score={best_f1_score:.4f}")
print(f"  For Recall: threshold={best_recall_threshold:.2f}, score={best_recall_score:.4f}")
print(f"  Default:    threshold=0.50")

# Visualize threshold impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot F1 vs threshold
thresholds = list(f1_scores.keys())
f1_values = list(f1_scores.values())
axes[0].plot(thresholds, f1_values, marker='o', linewidth=2)
axes[0].axvline(best_f1_threshold, color='red', linestyle='--', 
                label=f'Optimal: {best_f1_threshold:.2f}')
axes[0].axvline(0.5, color='gray', linestyle=':', label='Default: 0.50')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('F1 Score vs Threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot Recall vs threshold
recall_values = list(recall_scores.values())
axes[1].plot(thresholds, recall_values, marker='s', color='green', linewidth=2)
axes[1].axvline(best_recall_threshold, color='red', linestyle='--',
                label=f'Optimal: {best_recall_threshold:.2f}')
axes[1].axvline(0.5, color='gray', linestyle=':', label='Default: 0.50')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Recall')
axes[1].set_title('Recall vs Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Apply optimal threshold
preds_tuned = (probs_weighted >= best_f1_threshold).astype(int)

print("\n" + "="*60)
print("RESULTS WITH TUNED THRESHOLD")
print("="*60)
metrics_tuned = compute_metrics(labels_val, preds_tuned, probs_weighted, print_report=True)

print("\nComparison (Weighted CE):")
print(f"                Default (0.50)    Tuned ({best_f1_threshold:.2f})    Improvement")
print("-" * 65)
print(f"F1 Score:       {metrics_weighted['f1']:.4f}            "
      f"{metrics_tuned['f1']:.4f}         {metrics_tuned['f1'] - metrics_weighted['f1']:+.4f}")
print(f"Recall:         {metrics_weighted['recall']:.4f}            "
      f"{metrics_tuned['recall']:.4f}         {metrics_tuned['recall'] - metrics_weighted['recall']:+.4f}")
print(f"Precision:      {metrics_weighted['precision']:.4f}            "
      f"{metrics_tuned['precision']:.4f}         {metrics_tuned['precision'] - metrics_weighted['precision']:+.4f}")
print("="*60)

## Part 8: Comprehensive Comparison

### Summary of All Approaches:

Let's compare all methods side-by-side on the test set.

In [ ]:
print("\n" + "="*60)
print("FINAL EVALUATION ON TEST SET")
print("="*60)

# Get test predictions for all models
def get_test_predictions(model, test_loader, threshold=0.5):
    """Get predictions on test set."""
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            outputs = model(features)
            probs = F.softmax(outputs, dim=-1)[:, 1]  # Prob of class 1
            preds = (probs >= threshold).long()
            
            all_preds.append(preds.cpu())
            all_probs.append(probs.cpu())
            all_labels.append(labels)
    
    return (
        torch.cat(all_preds).numpy(),
        torch.cat(all_probs).numpy(),
        torch.cat(all_labels).numpy()
    )

# Evaluate all models
results = {}

print("\n1. Baseline (Standard CE, threshold=0.5):")
test_preds_baseline, test_probs_baseline, test_labels = get_test_predictions(
    model_baseline, test_loader, threshold=0.5
)
results['Baseline'] = compute_metrics(
    test_labels, test_preds_baseline, test_probs_baseline, print_report=False
)

print("\n2. Weighted CE (threshold=0.5):")
test_preds_weighted, test_probs_weighted, _ = get_test_predictions(
    model_weighted, test_loader, threshold=0.5
)
results['Weighted CE'] = compute_metrics(
    test_labels, test_preds_weighted, test_probs_weighted, print_report=False
)

print("\n3. Weighted CE + Tuned Threshold:")
test_preds_tuned, test_probs_tuned, _ = get_test_predictions(
    model_weighted, test_loader, threshold=best_f1_threshold
)
results['Weighted + Tuned'] = compute_metrics(
    test_labels, test_preds_tuned, test_probs_tuned, print_report=False
)

print("\n4. Focal Loss (threshold=0.5):")
test_preds_focal, test_probs_focal, _ = get_test_predictions(
    model_focal, test_loader, threshold=0.5
)
results['Focal Loss'] = compute_metrics(
    test_labels, test_preds_focal, test_probs_focal, print_report=False
)

# Create comparison table
print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON - TEST SET")
print("="*80)
print(f"{'Method':<25} {'Precision':>10} {'Recall':>10} {'F1':>10} {'PR-AUC':>10}")
print("-" * 80)

for name, metrics in results.items():
    print(f"{name:<25} {metrics['precision']:>10.4f} {metrics['recall']:>10.4f} "
          f"{metrics['f1']:>10.4f} {metrics['pr_auc']:>10.4f}")

print("="*80)

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

methods = list(results.keys())
metrics_to_plot = ['precision', 'recall', 'f1', 'pr_auc']
titles = ['Precision', 'Recall', 'F1 Score', 'PR-AUC']

for idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    row, col = idx // 2, idx % 2
    values = [results[method][metric] for method in methods]
    
    colors = ['red', 'orange', 'lightgreen', 'green']
    axes[row, col].barh(methods, values, color=colors, alpha=0.7)
    axes[row, col].set_xlabel(title)
    axes[row, col].set_xlim(0, 1.0)
    axes[row, col].grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, v in enumerate(values):
        axes[row, col].text(v + 0.02, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("KEY INSIGHTS:")
print("="*80)
print("1. Baseline fails: High accuracy but terrible recall")
print("2. Weighted CE: Significant improvement in recall and F1")
print("3. Threshold tuning: Further boosts target metric")
print("4. Focal Loss: Excellent for this imbalance ratio")
print("\nBest overall: Weighted CE + Tuned Threshold or Focal Loss")
print("Choice depends on: precision vs recall trade-off for your use case")
print("="*80)

## 📊 Summary

### Techniques for Imbalanced Classification:

**1. Standard Cross-Entropy (Baseline)**
- ❌ Fails for imbalanced data
- ❌ High accuracy but low recall
- ❌ Model ignores minority class
- Use: Never for imbalanced problems

**2. Weighted Cross-Entropy**
- ✓ Simple to implement
- ✓ Significant improvement
- ✓ Works well for moderate imbalance
- Use: Imbalance ratio 5:1 to 100:1

**3. Focal Loss**
- ✓ Best for extreme imbalance
- ✓ Focuses on hard examples
- ✓ Self-adjusting
- Use: Imbalance ratio >100:1

**4. Threshold Tuning**
- ✓ Optimize for your specific metric
- ✓ Business-driven decisions
- ✓ Easy to implement
- Use: Always, after training

### Decision Guide:

```
Imbalance Ratio?
  |
  ├─ < 5:1 → Standard CE + Threshold Tuning
  ├─ 5:1 to 20:1 → Weighted CE + Threshold Tuning
  ├─ 20:1 to 100:1 → Weighted CE or Focal Loss
  └─ > 100:1 → Focal Loss + Sampling

Optimization Goal?
  |
  ├─ Maximize Recall → Low threshold (e.g., 0.2)
  ├─ Maximize Precision → High threshold (e.g., 0.7)
  ├─ Balance Both → Tune for F1
  └─ Business Costs → Cost-based threshold
```

### Evaluation Metrics Checklist:

✓ **Never rely on accuracy alone**
✓ **Always report precision and recall**
✓ **Use F1 for balanced view**
✓ **Use PR-AUC over ROC-AUC**
✓ **Show confusion matrix**
✓ **Report per-class metrics**

### Common Pitfalls:

**Pitfall 1: Trusting accuracy**
```
99% accuracy → Sounds great!
Actually: Predicts majority class for everything
Solution: Check recall for minority class
```

**Pitfall 2: Not tuning threshold**
```
Using default 0.5 threshold
Actually: Arbitrary, often suboptimal
Solution: Grid search for optimal threshold
```

**Pitfall 3: Wrong evaluation metric**
```
Optimizing for accuracy or ROC-AUC
Actually: Misleading for imbalanced data
Solution: Use F1, PR-AUC, or business-specific metric
```

**Pitfall 4: Ignoring business context**
```
Treating all errors equally
Actually: Missing fraud != False alarm
Solution: Cost-sensitive learning
```

### Production Deployment:

**Before deployment**:
1. Validate on held-out test set
2. Check performance on different subgroups
3. A/B test threshold values
4. Monitor performance over time
5. Set up alerts for metric degradation

**In production**:
1. Log all predictions and probabilities
2. Collect ground truth labels when available
3. Retrain periodically (data drift)
4. Adjust threshold based on business feedback
5. Monitor minority class performance specifically

### Real-World Examples:

| Domain | Imbalance | Best Approach |
|--------|-----------|---------------|
| Credit Card Fraud | 0.1-1% fraud | Focal Loss + Low Threshold |
| Medical Diagnosis | 1-5% disease | Weighted CE + Recall Optimization |
| Manufacturing Defects | 0.5-2% defective | Focal Loss + Cost-Based Threshold |
| Click-Through Rate | 1-3% clicks | Weighted CE + Calibration |
| Spam Detection | 5-10% spam | Weighted CE + Precision Focus |

### Next Steps:

**Notebook 15: Mixed Precision Training**
- FP16/BF16 for memory savings
- Automatic Mixed Precision (AMP)
- Gradient scaling
- Numerical stability with custom losses

**Advanced Techniques** (Stage 2):
- SMOTE and data augmentation
- Ensemble methods
- Anomaly detection approaches
- Online learning for changing distributions

---

**Key Takeaway**: Imbalanced classification is NOT solved by accuracy alone. Use proper metrics (F1, PR-AUC), appropriate loss functions (weighted CE or focal loss), and threshold tuning. Always validate that your model actually catches the minority class!